Script to parse DBLP anthology. Results of the parsing saved in pickle format for faster loading.

**Pre-requisites:**

1. Download the DBLP dump from November 2026 (`dblp-2026-11-01.xml.gz`) from [DROPS](https://drops.dagstuhl.de/entities/artifact/10.4230/dblp.xml.2026-11-01).
2. Download the corresponding `.dtd` file (`dblp-2023-06-28.dtd`) from [DROPS](https://drops.dagstuhl.de/entities/artifact/10.4230/dblp.xml.dtd.2023-06-28).
3. Save both files in the `data/` folder.

In [39]:
from pathlib import Path
import contextlib
import gzip
import re
from lxml import etree
import pyarrow as pa
import pyarrow.parquet as pq
import pandas as pd
from tqdm.auto import tqdm

In [40]:
DATA_DIR = Path("../data").resolve()
XML_PATH = DATA_DIR / "dblp-2026-11-01.xml.gz"
DTD_PATH = DATA_DIR / "dblp-2023-06-28.dtd"
OUT_PATH = DATA_DIR / "dblp_anthology.parquet"

START_YEAR = 2020
END_YEAR = 2026
CHUNK_SIZE = 200_000  # records per write to Parquet

In [41]:
assert XML_PATH.exists(), f"Missing {XML_PATH}"
assert DTD_PATH.exists(), f"Missing {DTD_PATH} (must be next to the XML dump)"

if XML_PATH.suffix == ".gz":
    with open(XML_PATH, "rb") as f:
        assert f.read(2) == b"\x1f\x8b", "Not a gzip file - possibly the bot-check HTML page"

print(f"Dump: {XML_PATH.stat().st_size / 1e9:.2f} GB")

Dump: 1.12 GB


In [42]:
# Same type labels as the DBLP search API
RECORD_TYPES = {
    "article": "Journal Articles",
    "inproceedings": "Conference and Workshop Papers",
    "incollection": "Parts in Books or Collections",
}

SCHEMA = pa.schema([
    ("dblp_key", pa.string()),
    ("title", pa.string()),
    ("authors", pa.list_(pa.string())),
    ("year", pa.int16()),
    ("venue", pa.string()),
    ("type", pa.string()),
    ("doi", pa.string()),
    ("url", pa.string()),
])

def text_of(elem):
    # Titles can contain markup like <i> or <sub>, so join all inner text
    return "".join(elem.itertext()).strip() if elem is not None else None

def parse_record(elem):
    year = elem.findtext("year")
    if not year or not year.isdigit() or int(year) < START_YEAR or int(year) > END_YEAR:
        return None

    title = text_of(elem.find("title"))
    if not title:
        return None

    ees = [e.text for e in elem.findall("ee") if e.text]
    doi = next((e.split("doi.org/", 1)[1] for e in ees if "doi.org/" in e), None)
    key = elem.get("key")
    
    # Removing preprint and arXiv entries
    is_informal = elem.get("publtype") == "informal" or key.startswith("journals/corr/")
    if is_informal:
        return None

    return {
        "dblp_key": key,
        "title": title.rstrip("."),
        "authors": [a.text for a in elem.findall("author") if a.text],
        "year": int(year),
        "venue": elem.findtext("journal") or elem.findtext("booktitle"),
        "type": RECORD_TYPES[elem.tag],
        "doi": doi,
        "url": f"https://dblp.org/rec/{key}",
    }

In [43]:
writer = pq.ParquetWriter(OUT_PATH, SCHEMA, compression="zstd")
buffer, n_scanned, n_kept = [], 0, 0

def flush(buf):
    writer.write_table(pa.Table.from_pylist(buf, schema=SCHEMA))

try:
    # Run from DATA_DIR so the parser finds the DTD named in the file header
    with contextlib.chdir(DATA_DIR), gzip.open(XML_PATH.name, "rb") as f:
        context = etree.iterparse(
            f,
            events=("end",),
            tag=tuple(RECORD_TYPES),
            load_dtd=True,          # reads the DTD from DATA_DIR
            resolve_entities=True,  # turns &uuml; etc. into real characters
            huge_tree=True,
        )

        for _, elem in tqdm(context, desc="Parsing DBLP", unit=" rec", mininterval=1):
            n_scanned += 1
            rec = parse_record(elem)
            if rec:
                buffer.append(rec)

            # Free memory: clear this record and drop already-processed siblings
            elem.clear()
            while elem.getprevious() is not None:
                del elem.getparent()[0]

            if len(buffer) >= CHUNK_SIZE:
                flush(buffer)
                n_kept += len(buffer)
                buffer = []
                print(f"{n_scanned:,} scanned, {n_kept:,} kept")

    if buffer:
        flush(buffer)
        n_kept += len(buffer)
finally:
    writer.close()

print(f"Done: {n_kept:,} papers from {START_YEAR} onwards saved to {OUT_PATH}")

Parsing DBLP: 627062 rec [00:14, 33461.19 rec/s]

651,338 scanned, 200,000 kept


Parsing DBLP: 1262808 rec [00:33, 37078.82 rec/s]

1,270,351 scanned, 400,000 kept


Parsing DBLP: 1962407 rec [00:50, 42666.17 rec/s]

1,985,246 scanned, 600,000 kept


... (18 outputs left out)

Parsing DBLP: 7272970 rec [03:43, 34182.35 rec/s]

7,272,970 scanned, 2,600,000 kept


Parsing DBLP: 8566456 rec [04:17, 33276.18 rec/s]


Done: 2,713,161 papers from 2020 onwards saved to /home/tom/Hallucination-KG-Survey/data/dblp_anthology.parquet


In [44]:
df = pd.read_parquet(OUT_PATH)

In [45]:
print(df["year"].value_counts().sort_index())

year
2020    354213
2021    376839
2022    391373
2023    415673
2024    449662
2025    438796
2026    286605
Name: count, dtype: int64


In [46]:
print(df["type"].value_counts())

type
Journal Articles                  1460010
Conference and Workshop Papers    1248119
Parts in Books or Collections        5032
Name: count, dtype: int64


In [47]:
print(df.iloc[0].to_string())

dblp_key                         reference/sp/KoprinskaY22
title             People-to-People Reciprocal Recommenders
authors                    [Irena Koprinska, Kalina Yacef]
year                                                  2022
venue                         Recommender Systems Handbook
type                         Parts in Books or Collections
doi                           10.1007/978-1-0716-2197-4_11
url         https://dblp.org/rec/reference/sp/KoprinskaY22
